In [1]:
import os
from pathlib import Path

import pandas as pd

from poc.data.bronze.dataset import MarketPriceDataset, GridStateDataset, WeatherDataset
from poc.data.bronze.schema import MarketPriceSchema, GridStateSchema, WeatherSchema
from poc.data.bronze.metadata import BronzeBaseDatasetMetadata

# set the start / end timestamp (in day format) for a given dataset to download it
start = pd.Timestamp("2024-10-01", tz="UTC")
end   = pd.Timestamp("2024-10-02", tz="UTC")


In [2]:
__ROOT = Path(os.getcwd())

In [3]:
# load the meta-data (from *.yaml file; pydantic-validated)
market_price_metadata = BronzeBaseDatasetMetadata.from_yaml(
    str(__ROOT / "assets/datasets/market_prices.yaml"),
    schema=MarketPriceSchema
)

grid_state_metadata = BronzeBaseDatasetMetadata.from_yaml(
    str(__ROOT / "assets/datasets/grid_state.yaml"),
    schema=GridStateSchema,
)

In [4]:
# define dataset objects - stupid names, those classes are more loaders than dataset, but let's ignore it for now...
market_price_dataset = MarketPriceDataset(
    metadata=market_price_metadata,
    start=start,
    end=end,
)

grid_state_dataset = GridStateDataset(
    metadata=grid_state_metadata,
    api_key="b25938c6-f2d3-4eda-aef2-8d71580afdcc",
    start=start,
    end=end,
    country_code="ES",
)


In [5]:
# load datasets for a given timestamp ranges
market_df = market_price_dataset.load()
grid_state_df = grid_state_dataset.load()


#### MARKET DATASET

In [6]:
market_df.head()

,snapshot,issue_time,valid_time,data_source,data_type,market,price,currency,energy_unit,exchange_rate_to_pln
0,2026-03-16 22:09:08.012374+00:00,2024-09-30 13:00:00+00:00,2024-09-30 22:00:00+00:00,OMIE,OBSERVATION,ID1,103.75,EUR,MWH,1.0
1,2026-03-16 22:09:08.012374+00:00,2024-09-30 13:00:00+00:00,2024-09-30 22:15:00+00:00,OMIE,OBSERVATION,ID1,98.18,EUR,MWH,1.0
2,2026-03-16 22:09:08.012374+00:00,2024-09-30 13:00:00+00:00,2024-09-30 22:30:00+00:00,OMIE,OBSERVATION,ID1,97.99,EUR,MWH,1.0
3,2026-03-16 22:09:08.012374+00:00,2024-09-30 13:00:00+00:00,2024-09-30 22:45:00+00:00,OMIE,OBSERVATION,ID1,95.36,EUR,MWH,1.0
4,2026-03-16 22:09:08.012374+00:00,2024-09-30 13:00:00+00:00,2024-09-30 23:00:00+00:00,OMIE,OBSERVATION,ID1,84.51,EUR,MWH,1.0


Market data is a dataset that contains the time series for prices on (SPANISH) energy market. Unfortunately only Spanish / Portugal energy markets publishes full historical data for free.
The most important column to understand here is the `market` column. There are 4 types of `market` phases of shaping the prices.
- day ahead market (DAM): prices that forms in the previous day around 12:00 (noon)
- intraday market phase 1 (ID1): prices that forms in the previous day (around evening)
- intraday market phase 2 & 3 (ID2, ID3): prices that forms in the current day

Column `issue_time` reflect the time, when a given price was formed and publish by the exchange. Column `valid time` describe for which timestamp given price was formed. For example; if we have a price `100 EUR` for `issue_time = 12:00`, and `valid_time = 15:00` it means, that
energy price was `100 EUR` on `15:00`, but the price was published by the exchange (the price from current stage overrides the price from the previous market stage; exception is DAM - it is the first stage). Energy trader can reason as follows: he might allocate some energy to be sold
at `15:00` knowing DAM prices, than, after the update, he might allocate another portion of energy to be sold in other timestamp (he have generation predictions, if he have PV or wind generation, or just plan how to utilize the energy stored in the energy storage); basically, the
idea is that trader must follow his intuition to allocate some energy knowing only the IDM market prices, but left some to allocate when he will get the "price update" (ID1, ID2, ID3). The decision he makes also depend on various "energetic" constraints, here is a simple example of
such constraints: https://www.twaice.com/best-practices/battery-aging-the-overlooked-factor-in-bess-trading-strategies

NOTE (THIS WILL BE THE CORE FUN IN THIS PROJECT): the aim for the prediction model is not necessarily to predict energy prices (for ID market stages; DA market is always known apriori, so DA market prices is the most important feature, I believe). The goal is to somehow quantify the trader dillema (but in the next stages) to
determine, what is the best "fraction" of energy that is left, to use knowing what is already published, and what fraction of energy should be saved for the next market phase. To be able to train the model, I think we will need to sync the training of the model (or the loss funcion)
with the optimization problem, that will model all the physical constraint of the problem (like the aging / degradation of the energy storage as it is being exploited in time, or other constraints that model PV or wind farms).

#### GRID STATE DATASET

In [7]:
grid_state_df.head()

,snapshot,issue_time,valid_time,data_source,data_type,system_load,solar_generation,wind_offshore_generation,wind_onshore_generation,power_unit
0,2026-03-16 22:09:10.706565+00:00,2026-03-16 22:09:10.706565+00:00,2024-10-01 00:00:00+00:00,ENTSOE,OBSERVATION,22132.0,552.0,0.0,5000.0,MW
1,2026-03-16 22:09:10.706565+00:00,2026-03-16 22:09:10.706565+00:00,2024-10-01 00:15:00+00:00,ENTSOE,OBSERVATION,21812.0,552.0,0.0,5056.0,MW
2,2026-03-16 22:09:10.706565+00:00,2026-03-16 22:09:10.706565+00:00,2024-10-01 00:30:00+00:00,ENTSOE,OBSERVATION,21608.0,548.0,0.0,5100.0,MW
3,2026-03-16 22:09:10.706565+00:00,2026-03-16 22:09:10.706565+00:00,2024-10-01 00:45:00+00:00,ENTSOE,OBSERVATION,21544.0,548.0,0.0,5200.0,MW
4,2026-03-16 22:09:10.706565+00:00,2026-03-16 22:09:10.706565+00:00,2024-10-01 01:00:00+00:00,ENTSOE,OBSERVATION,21360.0,548.0,0.0,5240.0,MW


This dataset is just the set of initial features. I believe it should be enough features to create first simple prediction model.